# 🃏 MTG Oracle — Fine-tuning + RAG Training Notebook

This notebook fine-tunes **microsoft/phi-2** (2.8B parameters) on Magic: The Gathering rulings
using **LoRA** (Low-Rank Adaptation) for memory-efficient training.

The trained model is later combined with a **RAG system** (ChromaDB + Scryfall data)
to produce accurate, grounded responses about MTG rules and card evaluations.

## Architecture
```
Scryfall API → Dataset (76,716 Q&A pairs) → Fine-tuning (LoRA) → phi-2 specialized model
                                                                         ↓
                                              User query → ChromaDB RAG → Fine-tuned model → Answer
```

## Requirements
- GPU runtime (T4 or better) — Runtime → Change runtime type → T4 GPU
- ~15GB disk space for model weights and dataset
- Hugging Face account for model upload

## 1. Setup & Dependencies

In [ ]:
# Verify GPU is available — required for training
# Go to Runtime → Change runtime type → T4 GPU if this returns False
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None — switch to GPU runtime!")

In [ ]:
# Install required libraries
# - transformers: Hugging Face model loading and training
# - datasets: efficient dataset handling
# - peft: LoRA and other parameter-efficient fine-tuning methods
# - accelerate: hardware-agnostic training (GPU/TPU/CPU)
# - bitsandbytes: model quantization to reduce memory usage
!pip install -q transformers datasets peft accelerate bitsandbytes
!pip install -q requests

## 2. Data Collection — Scryfall API

Scryfall is the most comprehensive Magic: The Gathering database with a free public API.
We use their bulk data downloads to get all rulings efficiently.

**Why bulk download instead of individual requests?**
- 37,000+ cards × 2 requests each = 74,000+ API calls
- Bulk download = 2 requests total
- Scryfall explicitly recommends bulk downloads for large datasets

In [ ]:
import requests
import json
import time
from urllib.parse import quote

# Scryfall requires a custom User-Agent to identify your application
# Without this header, all requests return 400 Bad Request
headers = {
    "User-Agent": "MTG-Oracle-FineTuning/1.0 (educational project)",
    "Accept": "application/json"
}

# Test the API connection with a single card
card_name = "Snapcaster Mage"
url = f"https://api.scryfall.com/cards/named?fuzzy={quote(card_name)}"

response = requests.get(url, headers=headers)
print("Status:", response.status_code)

if response.status_code == 200:
    data = response.json()
    print("Card:", data.get('name'))
    print("Type:", data.get('type_line'))
    
    # Each card has a rulings_uri pointing to its official rulings
    rulings = requests.get(data['rulings_uri'], headers=headers).json()
    print(f"Rulings count: {len(rulings.get('data', []))}")
    for r in rulings.get('data', [])[:3]:  # show first 3 rulings
        print("-", r['comment'][:120])
else:
    print("Error:", response.json())

In [ ]:
# Download bulk data — much more efficient than individual requests
# Scryfall provides daily snapshots of their entire database

bulk_url = "https://api.scryfall.com/bulk-data"
response = requests.get(bulk_url, headers=headers)
bulk_data = response.json()

# Show available dumps and their sizes
print("Available bulk data dumps:")
for item in bulk_data['data']:
    print(f"  {item['type']:20} — {item['name']:25} — {item['size']//1024//1024} MB")

In [ ]:
# Download the two datasets we need:
# 1. rulings: all official rulings for all cards (24MB)
# 2. oracle_cards: card names, types, and oracle text (165MB)
# Both share 'oracle_id' as the join key

rulings_dump = next(item for item in bulk_data['data'] if item['type'] == 'rulings')
oracle_dump  = next(item for item in bulk_data['data'] if item['type'] == 'oracle_cards')

print("Downloading rulings dump (24MB)...")
all_rulings = requests.get(rulings_dump['download_uri'], headers=headers).json()
print(f"✓ Total rulings downloaded: {len(all_rulings)}")

print("\nDownloading oracle cards dump (165MB)...")
all_cards = requests.get(oracle_dump['download_uri'], headers=headers).json()
print(f"✓ Total cards downloaded: {len(all_cards)}")

## 3. Dataset Construction

We convert raw rulings into Q&A pairs in ChatML format — the standard for fine-tuning chat models.

Each training example has three roles:
- **system**: defines the model's persona and behavior (constant across all examples)
- **user**: the question (what users will ask at inference time)
- **assistant**: the correct answer (what the model learns to generate)

Two types of examples:
1. **Ruling pairs** (automated, from Scryfall) — factual rules interactions
2. **Evaluation pairs** (hand-crafted) — subjective card assessments with expert style

In [ ]:
# Join rulings with card data using oracle_id as the key
# This is equivalent to a SQL JOIN between two tables

# Index cards by oracle_id for O(1) lookup
cards_by_id = {}
for card in all_cards:
    oracle_id = card.get('oracle_id')
    if oracle_id:
        cards_by_id[oracle_id] = {
            "name":        card.get('name', ''),
            "type_line":   card.get('type_line', ''),
            "oracle_text": card.get('oracle_text', ''),
            "mana_cost":   card.get('mana_cost', ''),
        }

print(f"✓ Cards indexed: {len(cards_by_id)}")

# Group rulings by oracle_id (one card can have multiple rulings)
rulings_by_card = {}
for ruling in all_rulings:
    oracle_id = ruling.get('oracle_id')
    if oracle_id:
        if oracle_id not in rulings_by_card:
            rulings_by_card[oracle_id] = []
        rulings_by_card[oracle_id].append(ruling['comment'])

print(f"✓ Cards with at least one ruling: {len(rulings_by_card)}")

In [ ]:
# Build Q&A pairs from rulings
# Each ruling becomes one training example in ChatML format

SYSTEM_PROMPT = (
    "You are MTG Oracle, an expert Magic: The Gathering judge and analyst. "
    "You explain rules interactions clearly and evaluate cards with authority. "
    "For rulings, cite the relevant game rule when possible. "
    "For evaluations, give a score from 1-5 and justify it concisely."
)

training_pairs = []
skipped = 0

for oracle_id, rulings in rulings_by_card.items():
    card = cards_by_id.get(oracle_id)
    if not card:
        skipped += 1
        continue

    card_name = card['name']
    type_line = card['type_line']

    for ruling in rulings:
        # Skip rulings that are too short to be informative
        if len(ruling) < 30:
            continue

        pair = {
            "messages": [
                {"role": "system",    "content": SYSTEM_PROMPT},
                {"role": "user",      "content": f"What is the ruling for {card_name} ({type_line}) regarding its interaction with: {ruling[:80]}?"},
                {"role": "assistant", "content": f"Regarding {card_name}: {ruling}"}
            ]
        }
        training_pairs.append(pair)

print(f"✓ Ruling Q&A pairs created: {len(training_pairs)}")
print(f"  Skipped (no card data): {skipped}")

In [ ]:
# Hand-crafted evaluation examples
# These teach the model the subjective evaluation style — score + justification
# Fine-tuning learns STYLE from these examples, not just facts

card_evaluations = [
    {"card": "Snapcaster Mage",       "format": "Modern",    "evaluation": "5/5 — One of the best creatures ever printed. Two mana for a 2/1 flash that gives flashback to any instant or sorcery in your graveyard is absurdly efficient. It turns every spell in your deck into a two-for-one. Staple in every blue tempo and control deck in Modern and Legacy. If you're playing blue and not playing Snapcaster, you need a very good reason."},
    {"card": "Tarmogoyf",             "format": "Modern",    "evaluation": "4/5 — The king of efficient beaters has lost some ground in a faster format, but a 4/5 or 5/6 for two mana is still absurd. The problem is Modern got faster and Tarmogoyf doesn't have evasion or protection. Still a threat, but no longer the automatic 4-of it once was. Play it in Jund or BG Rock, not in aggressive decks that need to close faster."},
    {"card": "Oko, Thief of Crowns",  "format": "Modern",    "evaluation": "Banned/5 — Banned in literally every format for a reason. Three mana planeswalker that starts at 6 loyalty, makes food, turns any creature or artifact into a 3/3 elk, and gains loyalty by doing it. The design completely breaks the fundamental rules of card evaluation. There is no world in which this card is fair Magic."},
    {"card": "Blood Moon",            "format": "Modern",    "evaluation": "4/5 — One of the most format-warping cards in Modern. Turns all nonbasic lands into Mountains, which completely locks out three and four color decks that rely on fetchlands and shocklands. Best in UR or Mono-Red where you run basics anyway. The ceiling is 'opponent concedes on turn three', the floor is 'dead card against basics-heavy decks'. High risk, absurd reward."},
    {"card": "Force of Will",         "format": "Legacy",    "evaluation": "5/5 — The defining card of Legacy. Free counterspell that costs a blue card from hand. The fact that you can counter a spell on turn one for zero mana defines the entire format. Every combo deck has to respect Force of Will. Every blue deck plays four. Non-negotiable staple. The card that makes Legacy Legacy."},
    {"card": "Thassa's Oracle",       "format": "Commander", "evaluation": "5/5 in combo / 1/5 outside combo — A 1/3 for two blue mana that does nothing unless you have an empty library or near-empty library. But pair it with Demonic Consultation or Tainted Pact and you win the game instantly for two cards and two mana. One of the most efficient win conditions ever printed."},
    {"card": "Splinter Twin",         "format": "Modern",    "evaluation": "Banned/5 — Banned in Modern since 2016 and for good reason. Four mana enchantment that combos with Pestermite and Deceiver Exarch to make infinite hasty tokens at instant speed. The combo is two cards, both individually playable, and wins on turn four consistently."},
    {"card": "Emrakul, the Aeons Torn","format": "Legacy",   "evaluation": "5/5 as a reanimator target — 15 mana to hard cast but nobody hard casts Emrakul. Show and Tell or Reanimate it into play and your opponent takes an annihilator 6 trigger and then loses to a 15/15 flying trample protection from colored spells. The ultimate reanimator target."},
    {"card": "Isochron Scepter",      "format": "Commander", "evaluation": "3/5 standalone / 5/5 in combo — Two mana artifact that imprints an instant with CMC 2 or less and lets you copy it every turn for two mana. Pair it with Dramatic Reversal and infinite mana rocks and you get infinite mana."},
    {"card": "Sensei's Divining Top", "format": "Legacy",    "evaluation": "5/5 — The most powerful card selection engine ever printed at one mana. Look at the top three cards of your library and rearrange them. With counterbalance it locks opponents out of playing spells. Banned in Modern for slowing games to a crawl."},
    {"card": "Kitchen Finks",         "format": "Modern",    "evaluation": "4/5 — Three mana 3/2 that gains 2 life when it enters and persists back as a 2/1 that gains 2 more life. Four life total and two bodies for three mana is absurd value. Staple in Abzan Company and any deck running the persist combo."},
    {"card": "Jace, the Mind Sculptor","format": "Modern",   "evaluation": "5/5 — The most powerful planeswalker ever printed and it's not close. Four mana, starts at 3 loyalty, brainstorms for free every turn, bounces creatures, fateseals opponents, and the ultimate shuffles their library into their deck."},
]

# Convert evaluations to ChatML format
evaluation_pairs = []
for eval_data in card_evaluations:
    evaluation_pairs.append({
        "messages": [
            {"role": "system",    "content": SYSTEM_PROMPT},
            {"role": "user",      "content": f"Evaluate {eval_data['card']} for {eval_data['format']}."},
            {"role": "assistant", "content": eval_data['evaluation']}
        ]
    })

# Combine ruling pairs + evaluation pairs
training_pairs.extend(evaluation_pairs)
print(f"✓ Total training pairs: {len(training_pairs)}")
print(f"  Ruling pairs:     {len(training_pairs) - len(evaluation_pairs)}")
print(f"  Evaluation pairs: {len(evaluation_pairs)}")

# Save full dataset as JSONL (one JSON object per line)
with open('mtg_training_data_full.jsonl', 'w') as f:
    for pair in training_pairs:
        f.write(json.dumps(pair) + '\n')

print("\n✓ Dataset saved to mtg_training_data_full.jsonl")

In [ ]:
# Save dataset to Google Drive for persistence
# Colab sessions reset — files in /content/ are lost when session ends
# Google Drive persists between sessions

from google.colab import drive
drive.mount('/content/drive')

import shutil
shutil.copy('mtg_training_data_full.jsonl', '/content/drive/MyDrive/mtg_training_data_full.jsonl')
print("✓ Dataset saved to Google Drive")

## 4. Model Loading & LoRA Configuration

We use **microsoft/phi-2** (2.8B parameters) as the base model, fine-tuned with **LoRA**.

### Why LoRA?
Full fine-tuning updates all 2.8B parameters — requires 40GB+ VRAM.
LoRA adds small trainable matrices to specific layers:

```
Full fine-tuning: 2,800,000,000 trainable params → needs A100 GPU
LoRA fine-tuning:     2,621,440 trainable params → works on T4 GPU (0.09%)
```

### How LoRA works
Instead of modifying weight matrix W directly, LoRA adds two small matrices A and B:
```
output = (W + A×B) × input
         ↑           ↑
    frozen original  trainable LoRA adapter (tiny)
```
Only A and B are trained. W is never modified.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, TaskType

model_name = "microsoft/phi-2"
print(f"Loading {model_name}...")

# Load tokenizer — converts text to token IDs the model understands
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token  # phi-2 has no pad token, use EOS instead

# Load model in float16 — uses half the VRAM vs float32 with minimal quality loss
# device_map="auto" automatically places model on GPU
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,  # 2.8GB instead of 5.6GB VRAM
    device_map="auto",
    trust_remote_code=True      # phi-2 uses custom Microsoft code
)

print(f"✓ Model loaded — {sum(p.numel() for p in model.parameters())/1e9:.1f}B parameters")

In [ ]:
# Configure LoRA — defines which parts of the model to adapt
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,  # causal language modeling (next-token prediction)
    r=8,                           # rank: size of LoRA matrices (higher = more params, more capacity)
    lora_alpha=32,                 # scaling factor: controls how much LoRA influences outputs
    lora_dropout=0.1,              # regularization to prevent overfitting
    target_modules=["q_proj", "v_proj"]  # apply LoRA only to attention query and value projections
)

# Wrap the model with LoRA — freezes original weights, adds trainable adapters
model = get_peft_model(model, lora_config)

# Shows exactly how many parameters will be trained vs frozen
model.print_trainable_parameters()
# Expected output: trainable params: ~2.6M || all params: ~2.8B || trainable%: ~0.09

## 5. Dataset Preparation

Convert Q&A pairs from JSON format to tokenized tensors the model can process.

**Tokenization**: text → token IDs (numbers)
```
"Snapcaster Mage gives flashback" → [1234, 567, 890, 234, 789]
```

**Labels**: in causal LM, the model learns to predict the next token.
Labels are identical to input_ids but shifted one position — the model sees token N and predicts token N+1.

In [ ]:
import json
import random
from datasets import Dataset

# Load the full dataset
def load_jsonl(path):
    data = []
    with open(path, 'r') as f:
        for line in f:
            data.append(json.loads(line))
    return data

full_data = load_jsonl('mtg_training_data_full.jsonl')
print(f"✓ Loaded {len(full_data)} training pairs")

# Sample 5000 examples — enough for the model to learn, manageable on T4
# Training on all 76k examples would take ~40 hours on T4
random.seed(42)
sample_data = random.sample(full_data, 5000)
print(f"✓ Sampled {len(sample_data)} pairs for training")

In [ ]:
# Convert ChatML messages to plain text format phi-2 understands
def format_conversation(example):
    messages = example['messages']
    text = ""
    for msg in messages:
        if msg['role'] == 'system':
            text += f"System: {msg['content']}\n"
        elif msg['role'] == 'user':
            text += f"User: {msg['content']}\n"
        elif msg['role'] == 'assistant':
            text += f"Assistant: {msg['content']}\n"
    text += tokenizer.eos_token  # signal end of sequence
    return {"text": text}

# Convert messages to text format
dataset = Dataset.from_list(sample_data)
dataset = dataset.map(format_conversation)

# Tokenize — convert text to token IDs
def tokenize(example):
    result = tokenizer(
        example['text'],
        truncation=True,    # cut sequences longer than max_length
        max_length=512,     # max tokens per example (longer = more VRAM)
        padding='max_length' # pad shorter sequences to max_length for uniform batches
    )
    # Labels = input_ids for causal LM (predict next token)
    result['labels'] = result['input_ids'].copy()
    return result

tokenized_dataset = dataset.map(tokenize, remove_columns=dataset.column_names)

# 95% training, 5% evaluation
split = tokenized_dataset.train_test_split(test_size=0.05, seed=42)
train_dataset = split['train']
eval_dataset  = split['test']

print(f"✓ Train samples: {len(train_dataset)}")
print(f"✓ Eval samples:  {len(eval_dataset)}")

## 6. Fine-tuning

Training configuration is optimized for T4 GPU (16GB VRAM):

| Parameter | Value | Why |
|-----------|-------|-----|
| batch_size | 1 | T4 has limited VRAM |
| gradient_accumulation | 8 | Simulates batch size of 8 without extra memory |
| fp16 | True | Half precision = half VRAM |
| gradient_checkpointing | True | Recomputes activations instead of storing them |
| learning_rate | 2e-4 | Standard for LoRA fine-tuning |

**Effective batch size** = per_device_train_batch_size × gradient_accumulation_steps = 1 × 8 = 8

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # reduces GPU memory fragmentation

import torch
torch.cuda.empty_cache()  # free any leftover GPU memory

from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

training_args = TrainingArguments(
    output_dir="./mtg-oracle-v2",
    num_train_epochs=3,                   # pass through the dataset 3 times
    per_device_train_batch_size=1,        # 1 example at a time (VRAM constraint)
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,        # accumulate gradients for 8 steps before updating
    warmup_steps=50,                      # gradually increase LR for first 50 steps
    learning_rate=2e-4,                   # standard LoRA learning rate
    fp16=True,                            # float16 training — half the VRAM
    logging_steps=50,                     # log loss every 50 steps
    eval_strategy="epoch",                # evaluate after each epoch
    save_strategy="epoch",                # save checkpoint after each epoch
    load_best_model_at_end=True,          # keep the best checkpoint
    report_to="none",                     # disable wandb logging
    dataloader_pin_memory=False,
    gradient_checkpointing=True,          # trade speed for memory — recomputes activations
)

# Data collator for causal LM — handles padding and label masking
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False  # causal LM (not masked LM like BERT)
)

model.gradient_checkpointing_enable()

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
)

print("Starting training...")
print(f"  Train examples: {len(train_dataset)}")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"  Estimated time: 2-3 hours on T4\n")

# Training loop — updates LoRA weights to minimize prediction loss
# Watch the Training Loss decrease each epoch — that means the model is learning
trainer.train()

## 7. Save & Evaluate

Save the model locally and to Google Drive, then test it with sample queries.

**Note**: This saves only the LoRA adapter (~10MB), not the full phi-2 model.
To load the model later you need both phi-2 (base) and this adapter.

In [ ]:
# Save model and tokenizer locally
model.save_pretrained("./mtg-oracle-v2-final")
tokenizer.save_pretrained("./mtg-oracle-v2-final")
print("✓ Model saved locally")

# Backup to Google Drive (Colab resets between sessions)
from google.colab import drive
drive.mount('/content/drive')

import shutil
shutil.copytree(
    "./mtg-oracle-v2-final",
    "/content/drive/MyDrive/mtg-oracle-v2-final",
    dirs_exist_ok=True
)
print("✓ Model saved to Google Drive")

In [ ]:
# Test the fine-tuned model
# Note: without RAG, the model may hallucinate details
# The full system combines this model with ChromaDB for accurate responses

model.eval()  # disable dropout, freeze weights — inference mode

def ask_mtg_oracle(question, max_new_tokens=200):
    prompt = (
        "System: You are MTG Oracle, an expert Magic: The Gathering judge and analyst. "
        "You explain rules interactions clearly and evaluate cards with authority.\n"
        f"User: {question}\n"
        "Assistant:"
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():  # don't compute gradients during inference
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,        # 0=deterministic, 1=random — 0.7 is balanced
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response.split("Assistant:")[-1].strip()

# Test queries
tests = [
    "What happens when Snapcaster Mage gives flashback to a sorcery?",
    "Evaluate Blood Moon for Modern.",
    "How does the Splinter Twin combo work?",
]

for test in tests:
    print("=" * 60)
    print(f"Q: {test}")
    print("=" * 60)
    print(ask_mtg_oracle(test))
    print()

## 8. Upload to Hugging Face Hub

Upload the trained LoRA adapter to Hugging Face Hub so it can be used from any machine.

**Important**: This uploads the adapter only (~10MB), not phi-2.
To use the model: load phi-2 as base + load this adapter on top.

In [ ]:
# Login to Hugging Face using Colab Secrets
# Add your HF token in: left sidebar → key icon → Add new secret → name: HF_Token
from google.colab import userdata
from huggingface_hub import login

hf_token = userdata.get('HF_Token')
login(token=hf_token)
print("✓ Logged in to Hugging Face")

In [ ]:
from huggingface_hub import HfApi

api = HfApi()

# Create the repository on Hugging Face Hub
# exist_ok=True means no error if repo already exists
api.create_repo(
    repo_id="Razo3000/mtg-oracle",
    repo_type="model",
    exist_ok=True
)
print("✓ Repository created")

# Upload LoRA adapter and tokenizer to Hub
# Note: this uploads the adapter only, NOT phi-2 base model
model.push_to_hub("Razo3000/mtg-oracle")
tokenizer.push_to_hub("Razo3000/mtg-oracle")

print("\n✓ Model uploaded to Hugging Face")
print("View at: https://huggingface.co/Razo3000/mtg-oracle")
print("\nTo load this model:")
print("  base = AutoModelForCausalLM.from_pretrained('microsoft/phi-2')")
print("  model = PeftModel.from_pretrained(base, 'Razo3000/mtg-oracle')")

## Next Steps

This fine-tuned model is the LLM component of the full MTG Oracle system.

The complete pipeline combines:
1. **This model** (fine-tuned phi-2) — generates responses in expert MTG judge style
2. **ChromaDB** — stores 76,704 official Scryfall rulings as vector embeddings  
3. **RAG** — retrieves relevant rulings at query time, grounds responses in real data
4. **Streamlit** — user interface

```
User question
     ↓
ChromaDB finds top-5 relevant rulings (semantic search)
     ↓
Prompt: system + rulings context + user question
     ↓
Fine-tuned phi-2 generates grounded response
     ↓
Answer based on official Scryfall rulings
```

See the local project (`src/chain.py`, `src/ingest.py`, `app.py`) for the RAG implementation.